# ComfyUI + RealVisXL on Kaggle — Interactive GUI

Run ComfyUI in a Kaggle notebook and access its visual workflow editor from your browser.

## What this notebook sets up
- ComfyUI and the RealVisXL V4.0 SDXL checkpoint
- ComfyUI Manager, Impact Pack, Impact Subpack, and ControlNet Aux
- OpenPose SDXL ControlNet, DWPose, and YOLOX assets
- Basic file/node validation and ComfyUI startup
- A Cloudflare Quick Tunnel for interactive GUI access

## Before you start
1. Use a Kaggle notebook with **Internet enabled** and a compatible GPU accelerator.
2. Run cells 01–07 in order and check that validation passes.
3. Run cell 08 manually to create a temporary, email-protected Cloudflare Quick Tunnel. Enter the email address that should be allowed to access the GUI.
4. Open the printed URL in your browser. Keep the Kaggle session running while you use ComfyUI.

**Interactive use:** this notebook is designed for a live GUI session. Do not expect the GUI or tunnel to remain available after the Kaggle runtime ends. The Quick Tunnel URL is temporary.

**Security:** ComfyUI is a powerful interface. The tunnel cell uses Cloudflare email authentication; do not remove the access restriction when exposing the interface publicly. Never publish a live tunnel URL or personal data in notebook outputs.

**Model and code licenses:** this notebook downloads third-party software and model files from their respective repositories. Review and follow their licenses and terms. This notebook does not change those licenses.

**Reproducibility:** upstream repositories and package versions can change. If a dependency update breaks the notebook, inspect the logs and pin compatible versions rather than assuming the latest versions are interchangeable.


In [ ]:

# ============================================
# 01 — Bootstrap ComfyUI
# ============================================

from pathlib import Path
import subprocess
import sys

COMFY_DIR = Path("/kaggle/working/ComfyUI")
REPO_URL = "https://github.com/comfyanonymous/ComfyUI.git"

# 1. Clone ComfyUI only if the repository is absent.
if not (COMFY_DIR / "main.py").is_file():
    if COMFY_DIR.exists():
        raise RuntimeError(
            f"{COMFY_DIR} exists but main.py is missing. "
            "Inspect or remove the incomplete directory before retrying."
        )

    print("Cloning ComfyUI...")
    subprocess.run(
        ["git", "clone", REPO_URL, str(COMFY_DIR)],
        check=True,
    )
else:
    print("Existing ComfyUI installation found.")

# 2. Confirm the repository has its requirements file.
REQUIREMENTS = COMFY_DIR / "requirements.txt"

if not REQUIREMENTS.is_file():
    raise FileNotFoundError(
        f"ComfyUI requirements file not found: {REQUIREMENTS}"
    )

# 3. Install core dependencies.
print("Installing ComfyUI core dependencies...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(REQUIREMENTS),
    ],
    check=True,
)

# 4. Report the result.
print("\n" + "=" * 44)
print("COMFYUI BOOTSTRAP COMPLETE")
print("=" * 44)
print("Python:", sys.version.split()[0])
print("ComfyUI directory:", COMFY_DIR)
print("Main entry point:", COMFY_DIR / "main.py")
print("Requirements:", REQUIREMENTS)


In [ ]:

# ============================================
# 02 — Download RealVisXL V4.0
# ============================================

from pathlib import Path
import subprocess
import shutil

COMFY_DIR = Path("/kaggle/working/ComfyUI")
CHECKPOINT_DIR = COMFY_DIR / "models" / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = CHECKPOINT_DIR / "RealVisXL_V4.0.safetensors"

MODEL_URL = (
    "https://huggingface.co/SG161222/RealVisXL_V4.0/"
    "resolve/main/RealVisXL_V4.0.safetensors"
)

# 1. Confirm Cell 01 has completed.
if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("ComfyUI is missing. Run Cell 01 first.")

# 2. Install aria2 only if needed.
if shutil.which("aria2c") is None:
    print("Installing aria2...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "aria2", "-qq"],
        check=True,
    )

# 3. Download or resume the checkpoint.
# aria2's -c option continues a partial download when possible.
print("Downloading or resuming RealVisXL V4.0...")
subprocess.run(
    [
        "aria2c",
        "-c",
        "-x", "8",
        "-s", "8",
        "-k", "16M",
        "--file-allocation=none",
        "--summary-interval=10",
        "--console-log-level=notice",
        "-o", CHECKPOINT.name,
        "-d", str(CHECKPOINT_DIR),
        MODEL_URL,
    ],
    check=True,
)

# 4. Validate that the download is plausibly complete.
if not CHECKPOINT.is_file():
    raise RuntimeError("RealVisXL checkpoint was not created.")

size_bytes = CHECKPOINT.stat().st_size
size_gib = size_bytes / (1024 ** 3)

# The expected checkpoint is approximately 6.4 GiB.
if size_bytes < 6 * (1024 ** 3):
    raise RuntimeError(
        f"Checkpoint is suspiciously small ({size_gib:.2f} GiB). "
        "Check the download log and rerun this cell to resume."
    )

# 5. Report.
print("\n" + "=" * 44)
print("REALVISXL CHECKPOINT READY")
print("=" * 44)
print("File:", CHECKPOINT.name)
print("Path:", CHECKPOINT)
print(f"Size: {size_gib:.2f} GiB")
print("Status: Size check passed")


In [ ]:

# ============================================
# 03 — Install Custom Nodes & Dependencies
# ============================================

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import subprocess
import sys

COMFY_DIR = Path("/kaggle/working/ComfyUI")
CUSTOM_NODES = COMFY_DIR / "custom_nodes"

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("Run Cell 01 before installing custom nodes.")

CUSTOM_NODES.mkdir(parents=True, exist_ok=True)

REPOSITORIES = {
    "ComfyUI-Manager":
        "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-Impact-Pack":
        "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "ComfyUI-Impact-Subpack":
        "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git",
    "comfyui_controlnet_aux":
        "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
}

def clone_or_check(item):
    name, url = item
    destination = CUSTOM_NODES / name

    if destination.exists():
        if not (destination / ".git").is_dir():
            raise RuntimeError(
                f"{destination} exists but is not a Git repository. "
                "Inspect it before retrying."
            )
        if not (destination / "requirements.txt").is_file():
            print(f"[EXISTS] {name} — no requirements.txt at repository root")
        else:
            print(f"[EXISTS] {name}")
        return destination

    print(f"[CLONE] {name}")
    subprocess.run(
        ["git", "clone", "--depth", "1", url, str(destination)],
        check=True,
    )
    print(f"[DONE] {name}")
    return destination

# 1. Clone independent repositories concurrently.
with ThreadPoolExecutor(max_workers=4) as executor:
    repositories = list(
        executor.map(clone_or_check, REPOSITORIES.items())
    )

print("\nAll custom-node repositories are present.")

# 2. Collect dependency files so pip resolves them together.
requirements_files = [
    repo / "requirements.txt"
    for repo in repositories
    if (repo / "requirements.txt").is_file()
]

pip_command = [
    sys.executable, "-m", "pip", "install", "-q"
]

for requirements in requirements_files:
    pip_command.extend(["-r", str(requirements)])

# Additional packages used by the auxiliary/detection pipeline.
pip_command.extend(["ultralytics", "segment-anything"])

print(f"Installing dependencies from {len(requirements_files)} files...")
subprocess.run(pip_command, check=True)

# 3. Run the Impact Subpack installer if this version provides one.
subpack = CUSTOM_NODES / "ComfyUI-Impact-Subpack"
installer = subpack / "install.py"

if installer.is_file():
    print("Running Impact Subpack installer...")
    subprocess.run(
        [sys.executable, str(installer)],
        cwd=subpack,
        check=True,
    )
else:
    print("No Impact Subpack install.py found; requirements were installed.")

# 4. Report installed repositories.
print("\n" + "=" * 44)
print("CUSTOM NODE INSTALLATION COMPLETE")
print("=" * 44)

for name in REPOSITORIES:
    path = CUSTOM_NODES / name
    print(f"{name}: {'OK' if (path / '.git').is_dir() else 'MISSING'}")

print("Requirements files processed:", len(requirements_files))
print("Next: Cell 04 — download and validate auxiliary models.")


In [ ]:

# ============================================
# 04 — Download Auxiliary Models
# ============================================

from pathlib import Path
import subprocess
import shutil
import sys

COMFY_DIR = Path("/kaggle/working/ComfyUI")

# OpenPose ControlNet checkpoint
CONTROLNET_DIR = COMFY_DIR / "models" / "controlnet"
CONTROLNET_DIR.mkdir(parents=True, exist_ok=True)

CONTROLNET = CONTROLNET_DIR / "xinsir_openpose_sdxl_1.0.safetensors"
CONTROLNET_URL = (
    "https://huggingface.co/xinsir/"
    "controlnet-openpose-sdxl-1.0/resolve/main/"
    "diffusion_pytorch_model.safetensors"
)

# DWPose and YOLOX files used by comfyui_controlnet_aux
AUX_DIR = (
    COMFY_DIR / "custom_nodes" /
    "comfyui_controlnet_aux" / "ckpts"
)
AUX_DIR.mkdir(parents=True, exist_ok=True)

AUX_MODELS = [
    {
        "repo_id": "hr16/DWPose-TorchScript-BatchSize5",
        "filename": "dw-ll_ucoco_384_bs5.torchscript.pt",
    },
    {
        "repo_id": "hr16/yolox-onnx",
        "filename": "yolox_l.onnx",
    },
]

# 1. Ensure download tools are available.
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "aria2", "-qq"],
        check=True,
    )

# 2. Download or resume ControlNet.
print("Downloading/resuming OpenPose SDXL ControlNet...")
subprocess.run(
    [
        "aria2c", "-c",
        "-x", "8", "-s", "8", "-k", "16M",
        "--file-allocation=none",
        "--summary-interval=10",
        "-o", CONTROLNET.name,
        "-d", str(CONTROLNET_DIR),
        CONTROLNET_URL,
    ],
    check=True,
)

if not CONTROLNET.is_file() or CONTROLNET.stat().st_size < 100_000_000:
    raise RuntimeError("OpenPose ControlNet is missing or suspiciously small.")

print(
    f"ControlNet: {CONTROLNET.stat().st_size / (1024**3):.2f} GiB"
)

# 3. Download DWPose and YOLOX from Hugging Face.
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
        check=True,
    )
    from huggingface_hub import hf_hub_download

for item in AUX_MODELS:
    filename = item["filename"]
    destination = AUX_DIR / filename

    print(f"\nDownloading {filename}...")

    # Hugging Face caches downloads and supports resuming interrupted transfers.
    cached_path = hf_hub_download(
        repo_id=item["repo_id"],
        filename=filename,
    )

    # Avoid copying a file onto itself.
    cached_path = Path(cached_path)
    if cached_path.resolve() != destination.resolve():
        shutil.copy2(cached_path, destination)

    if not destination.is_file() or destination.stat().st_size == 0:
        raise RuntimeError(f"Model download failed validation: {filename}")

    print(f"Saved: {destination}")
    print(f"Size: {destination.stat().st_size / (1024**2):.1f} MiB")

# 4. Final report.
print("\n" + "=" * 44)
print("AUXILIARY MODEL DOWNLOADS COMPLETE")
print("=" * 44)

for path in [CONTROLNET] + [AUX_DIR / item["filename"] for item in AUX_MODELS]:
    print(f"{'OK' if path.is_file() and path.stat().st_size > 0 else 'MISSING'}: {path}")


In [ ]:

# ============================================
# 05 — Validate Files and Repositories
# ============================================

from pathlib import Path

COMFY_DIR = Path("/kaggle/working/ComfyUI")
CUSTOM_NODES = COMFY_DIR / "custom_nodes"

repositories = {
    "ComfyUI-Manager": CUSTOM_NODES / "ComfyUI-Manager",
    "ComfyUI-Impact-Pack": CUSTOM_NODES / "ComfyUI-Impact-Pack",
    "ComfyUI-Impact-Subpack": CUSTOM_NODES / "ComfyUI-Impact-Subpack",
    "comfyui_controlnet_aux": CUSTOM_NODES / "comfyui_controlnet_aux",
}

models = {
    "RealVisXL checkpoint":
        COMFY_DIR / "models/checkpoints/RealVisXL_V4.0.safetensors",
    "OpenPose SDXL ControlNet":
        COMFY_DIR / "models/controlnet/xinsir_openpose_sdxl_1.0.safetensors",
    "DWPose estimator":
        CUSTOM_NODES / "comfyui_controlnet_aux/ckpts/dw-ll_ucoco_384_bs5.torchscript.pt",
    "YOLOX detector":
        CUSTOM_NODES / "comfyui_controlnet_aux/ckpts/yolox_l.onnx",
}

print("=" * 50)
print("CUSTOM NODE REPOSITORIES")
print("=" * 50)

missing_repositories = []

for name, path in repositories.items():
    exists = (path / ".git").is_dir()
    print(f"{name}: {'OK' if exists else 'MISSING'}")
    if not exists:
        missing_repositories.append(name)

print("\n" + "=" * 50)
print("MODEL FILES")
print("=" * 50)

missing_models = []

for name, path in models.items():
    valid = path.is_file() and path.stat().st_size > 0
    size = f" ({path.stat().st_size / (1024**3):.2f} GiB)" if valid else ""
    print(f"{name}: {'OK' if valid else 'MISSING'}{size}")
    if not valid:
        missing_models.append(name)

print("\n" + "=" * 50)
print("IMPACT SUBPACK INSTALLATION FILES")
print("=" * 50)

subpack = repositories["ComfyUI-Impact-Subpack"]

for filename in ["install.py", "requirements.txt"]:
    path = subpack / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'NOT FOUND'}")

if missing_repositories or missing_models:
    print("\nVALIDATION FAILED")
    if missing_repositories:
        print("Missing repositories:", missing_repositories)
    if missing_models:
        print("Missing model files:", missing_models)
    raise RuntimeError("Fix missing assets before proceeding.")

print("\nFile and repository validation passed.")


In [ ]:

# ============================================
# 06 — Launch ComfyUI
# ============================================

from pathlib import Path
import subprocess
import time
import requests
import sys

COMFY_DIR = Path("/kaggle/working/ComfyUI")
LOG_FILE = Path("/kaggle/working/comfyui.log")
BASE_URL = "http://127.0.0.1:8188"

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("ComfyUI main.py not found. Run Cell 01.")

def is_ready():
    try:
        response = requests.get(f"{BASE_URL}/system_stats", timeout=3)
        return response.ok
    except requests.RequestException:
        return False

if is_ready():
    print("ComfyUI is already running.")
else:
    with open(LOG_FILE, "a", buffering=1) as log:
        process = subprocess.Popen(
            [
                sys.executable, "main.py",
                "--listen", "127.0.0.1",
                "--port", "8188",
            ],
            cwd=COMFY_DIR,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    print("Started ComfyUI. PID:", process.pid)
    print("Log file:", LOG_FILE)

    for _ in range(90):
        if is_ready():
            print("ComfyUI is ready.")
            break
        time.sleep(2)
    else:
        print("Recent log output:")
        if LOG_FILE.exists():
            print("\n".join(
                LOG_FILE.read_text(errors="replace").splitlines()[-60:]
            ))
        raise RuntimeError("ComfyUI failed to start.")


In [ ]:

# ============================================
# 07 — Validate Registered Nodes
# ============================================

import requests

BASE_URL = "http://127.0.0.1:8188"

response = requests.get(f"{BASE_URL}/object_info", timeout=60)
response.raise_for_status()
nodes = response.json()

required_nodes = {
    "Basic generation": [
        "CheckpointLoaderSimple",
        "CLIPTextEncode",
        "LoadImage",
        "VAEEncode",
        "KSampler",
        "VAEDecode",
        "SaveImage",
    ],
    "ControlNet": [
        "ControlNetLoader",
        "ControlNetApplyAdvanced",
    ],
    "DWPose": [
        "DWPreprocessor",
    ],
    "Impact Pack / Subpack": [
        "FaceDetailer",
        "UltralyticsDetectorProvider",
    ],
}

missing = []

for group, names in required_nodes.items():
    print(f"\n{group}")
    for name in names:
        registered = name in nodes
        print(f"  {name}: {'OK' if registered else 'MISSING'}")
        if not registered:
            missing.append(name)

if missing:
    print("\nSome required nodes are not registered:", missing)
    print("Inspect the ComfyUI startup log for import or dependency errors.")
    raise RuntimeError("Custom-node validation failed.")

print("\nAll required node types are registered.")


In [ ]:
# ============================================
# 08 — Start Email-Protected Cloudflare Quick Tunnel
# ============================================

from pathlib import Path
import subprocess
import requests
import time
import re
import platform
import urllib.request
import getpass

COMFY_URL = "http://127.0.0.1:8188"
LOG_FILE = Path("/kaggle/working/cloudflared.log")
BINARY = Path("/kaggle/working/cloudflared")

# 1. Verify ComfyUI is running.
response = requests.get(f"{COMFY_URL}/system_stats", timeout=10)
response.raise_for_status()
print("ComfyUI is responding locally.")

# 2. Ask privately which email address may access the public tunnel.
# The input is not echoed into the notebook output.
allowed_email = getpass.getpass(
    "Email address allowed to access this ComfyUI tunnel: "
).strip()
if not allowed_email or "@" not in allowed_email:
    raise ValueError("Enter a valid email address to protect the tunnel.")

# 3. Download cloudflared if it isn't already installed.
if not BINARY.exists():
    machine = platform.machine().lower()
    if machine in ("x86_64", "amd64"):
        architecture = "amd64"
    elif machine in ("aarch64", "arm64"):
        architecture = "arm64"
    else:
        raise RuntimeError(f"Unsupported architecture: {machine}")

    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-{architecture}"
    )
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(download_url, BINARY)
    BINARY.chmod(0o755)

print("cloudflared:", BINARY)

# 4. Start an email-protected Quick Tunnel.
# Anyone visiting the URL must pass Cloudflare email OTP authentication.
with open(LOG_FILE, "w", buffering=1) as log_handle:
    tunnel_process = subprocess.Popen(
        [
            str(BINARY),
            "tunnel",
            "--url", COMFY_URL,
            "--no-autoupdate",
            "--allowed-mail", allowed_email,
        ],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Started email-protected Cloudflare Tunnel. PID:", tunnel_process.pid)

# 5. Wait for the temporary public URL.
public_url = None
for _ in range(60):
    if tunnel_process.poll() is not None:
        break
    log_text = LOG_FILE.read_text(errors="replace") if LOG_FILE.exists() else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)

if not public_url:
    log_text = LOG_FILE.read_text(errors="replace") if LOG_FILE.exists() else ""
    print("Recent cloudflared log:")
    print(log_text[-5000:])
    raise RuntimeError("Cloudflare Tunnel did not provide a public URL.")

print("\n" + "=" * 44)
print("EMAIL-PROTECTED CLOUDFLARE TUNNEL READY")
print("=" * 44)
print("Public URL:", public_url)
print("Access requires an email one-time PIN.")
print("Keep the Kaggle session and tunnel process running.")


In [ ]:

# ============================================
# 09 — Final Pipeline Health Check
# ============================================

from pathlib import Path
import requests

COMFY_DIR = Path("/kaggle/working/ComfyUI")
BASE_URL = "http://127.0.0.1:8188"

# Check server.
response = requests.get(f"{BASE_URL}/system_stats", timeout=10)
response.raise_for_status()
print("ComfyUI server: ONLINE")

# Check model files.
model_paths = {
    "RealVisXL":
        COMFY_DIR / "models/checkpoints/RealVisXL_V4.0.safetensors",
    "OpenPose SDXL ControlNet":
        COMFY_DIR / "models/controlnet/xinsir_openpose_sdxl_1.0.safetensors",
    "DWPose":
        COMFY_DIR / "custom_nodes/comfyui_controlnet_aux/ckpts/dw-ll_ucoco_384_bs5.torchscript.pt",
    "YOLOX":
        COMFY_DIR / "custom_nodes/comfyui_controlnet_aux/ckpts/yolox_l.onnx",
}

print("\nModel files:")
missing_models = []

for name, path in model_paths.items():
    valid = path.is_file() and path.stat().st_size > 0
    print(f"{name}: {'OK' if valid else 'MISSING'}")
    if not valid:
        missing_models.append(name)

# Check node registration.
response = requests.get(f"{BASE_URL}/object_info", timeout=60)
response.raise_for_status()
nodes = response.json()

required_nodes = [
    "CheckpointLoaderSimple",
    "CLIPTextEncode",
    "LoadImage",
    "VAEEncode",
    "KSampler",
    "VAEDecode",
    "SaveImage",
    "ControlNetLoader",
    "ControlNetApplyAdvanced",
    "DWPreprocessor",
    "FaceDetailer",
    "UltralyticsDetectorProvider",
]

print("\nRegistered nodes:")
missing_nodes = []

for name in required_nodes:
    valid = name in nodes
    print(f"{name}: {'OK' if valid else 'MISSING'}")
    if not valid:
        missing_nodes.append(name)

if missing_models or missing_nodes:
    print("\nPIPELINE STATUS: NOT READY")
    if missing_models:
        print("Missing models:", missing_models)
    if missing_nodes:
        print("Missing nodes:", missing_nodes)
    raise RuntimeError("Resolve the missing components before generation.")

print("\nPIPELINE STATUS: ALL CHECKS PASSED")
